# TuiML Command-Line Interface (CLI)

Every TuiML capability is available from the terminal. The CLI, the Python API, and the MCP tools an agent calls are three faces of the same engine — `tuiml train` and `tuiml_train` take the same information.

| Group | Commands |
|-------|----------|
| **Core workflow** | `train`, `predict`, `evaluate`, `experiment`, `tune`, `test-statistics` |
| **Data** | `upload`, `read-data`, `profile`, `generate`, `preprocess`, `select-features`, `plot` |
| **Discovery** | `list`, `describe` |
| **Serving** | `serve`, `save`, `stop-server`, `status` |
| **Agent-authored algorithms** | `get-skeleton`, `create-algorithm`, `read-algorithm`, `edit-algorithm`, `delete-algorithm`, `list-files`, `search-source` |
| **System** | `setup`, `uninstall`, `info`, `update`, `restart`, `mcp`, `trace` |

Every command takes **named options** (`-a/--algorithm`, `-d/--data`, `-t/--target`) — no positional arguments — and every one accepts `--json-output` for scripting.

Run `tuiml <command> --help` for the full option list of any command.

## Installation & Setup

The CLI is included with TuiML. After installation, it's available as `tuiml`:

```bash
# Check installation
tuiml --version

# Show install method, package path, and whether an update is available
tuiml info

# Full command list
tuiml --help
```

## 1. Training Models (`tuiml train`)

Train models directly from the command line with preprocessing, feature selection, and evaluation.

### 1.1 Basic Training

```bash
# Simple training with default parameters and an 80/20 holdout
tuiml train -a RandomForestClassifier -d iris.csv -t class

# -d also accepts a built-in dataset name instead of a file path
tuiml train -a RandomForestClassifier -d iris -t class
```

Options are named, not positional: `-a/--algorithm`, `-d/--data`, `-t/--target`.

### 1.2 Choosing an Evaluation Strategy

```bash
# 10-fold cross-validation
tuiml train -a RandomForestClassifier -d iris.csv -t class --cv 10

# Holdout with a custom test proportion
tuiml train -a SVC -d data.csv -t target --test-size 0.3

# Pick the metrics explicitly - use the exact function names from
# tuiml.evaluation.metrics
tuiml train -a SVC -d data.csv -t target --cv 5 -m accuracy_score -m f1_score
```

### 1.3 Preprocessing and Feature Selection

```bash
# Add preprocessing steps, in order (exact class names)
tuiml train -a SVC -d data.csv -t target -p SimpleImputer -p StandardScaler

# Add a feature selection method
tuiml train -a SVC -d data.csv -t target -p StandardScaler -f SelectKBestSelector

# Or apply a whole preset in one flag
tuiml train -a RandomForestClassifier -d data.csv -t target --preset standard

# Available presets: minimal, fast, standard, full, imbalanced
```

### 1.4 Algorithm Parameters

```bash
# Hyperparameters as a JSON dict
tuiml train -a RandomForestClassifier -d iris.csv -t class \
    -P '{"n_estimators": 100, "max_depth": 10}'

# Everything together
tuiml train -a SVC -d data.csv -t target \
    -p StandardScaler \
    -f SelectKBestSelector \
    -P '{"kernel": "rbf", "C": 1.0}' \
    --cv 10 \
    --random-seed 42
```

Note the capital `-P` for algorithm parameters — lowercase `-p` is preprocessing.

### 1.5 Saving the Model and the Results

```bash
# Save metrics to a JSON file
tuiml train -a RandomForestClassifier -d iris.csv -t class --cv 10 -o results.json

# Save the trained model to a path you choose
tuiml train -a RandomForestClassifier -d iris.csv -t class --save-path ./model.pkl

# Verbose output for debugging
tuiml train -a RandomForestClassifier -d iris.csv -t class --cv 10 -v
```

Every run is persisted automatically. The output prints a **Model ID** and a **Model Path**; either one identifies the model for `predict`, `evaluate`, `serve`, and `save`.

The saved file is the **whole pipeline** — the preprocessing steps you passed with `-p`/`-f` travel with the model, so `tuiml predict` feeds it raw rows.

## 2. Making Predictions (`tuiml predict`)

Use a trained model to make predictions on new data.

### 2.1 Basic Prediction

```bash
# By file path
tuiml predict --model-path ./model.pkl -d test_data.csv

# Or by the Model ID that `tuiml train` printed
tuiml predict --model-id e9071dbe77d2 -d test_data.csv
```

### 2.2 Probabilities, Forecasts, and Output Files

```bash
# Save predictions to CSV
tuiml predict --model-path ./model.pkl -d test.csv -o predictions.csv

# Class probabilities instead of labels
tuiml predict --model-path ./model.pkl -d test.csv --stage predict_proba

# Time-series forecast n steps ahead
tuiml predict --model-path ./arima.pkl --stage forecast --steps 12

# Raw JSON, for piping into jq
tuiml predict --model-path ./model.pkl -d test.csv --json-output
```

## 3. Evaluating Models (`tuiml evaluate`)

Compute performance metrics for a trained model on test data.

### 3.1 Basic Evaluation

```bash
# Auto-detected metrics for the task
tuiml evaluate --model-path ./model.pkl -d test.csv -t class

# Or by Model ID
tuiml evaluate --model-id e9071dbe77d2 -d test.csv -t class
```

### 3.2 Specific Metrics and Reports

```bash
# Choose the metrics (exact function names)
tuiml evaluate --model-path ./model.pkl -d test.csv -t class \
    -m accuracy_score -m f1_score -m precision_score

# A full per-class classification report instead of a metric dict
tuiml evaluate --model-path ./model.pkl -d test.csv -t class --stage report

# Save results to JSON
tuiml evaluate --model-path ./model.pkl -d test.csv -t class -o evaluation.json
```

## 4. Running Experiments (`tuiml experiment`)

Compare multiple algorithms using cross-validation. `-a` and `-d` are repeatable.

### 4.1 Basic Experiment

```bash
# Compare three algorithms with 10-fold CV
tuiml experiment \
    -a RandomForestClassifier \
    -a SVC \
    -a NaiveBayesClassifier \
    -d iris.csv \
    -t class \
    --cv 10

# Across several datasets at once
tuiml experiment -a SVC -a RandomForestClassifier -d iris -d glass -t class
```

### 4.2 Exporting Results

```bash
# Markdown table (the default format)
tuiml experiment -a RandomForestClassifier -a SVC -d data.csv -t target \
    -o results.md -f markdown

# LaTeX, for papers
tuiml experiment -a RandomForestClassifier -a SVC -d data.csv -t target \
    -o table.tex -f latex

# CSV or JSON, for further analysis
tuiml experiment -a RandomForestClassifier -a SVC -d data.csv -t target -o results.csv -f csv
tuiml experiment -a RandomForestClassifier -a SVC -d data.csv -t target -o results.json -f json
```

### 4.3 Statistical Comparison

```bash
# Generate a critical difference plot alongside the table
tuiml experiment \
    -a RandomForestClassifier -a SVC -a NaiveBayesClassifier -a C45TreeClassifier \
    -d iris.csv -t class \
    --cv 10 \
    --plot
```

The command already prints pairwise p-values and flags which differences are significant. For a formal omnibus test plus post-hoc, feed the per-fold scores to `tuiml test-statistics`:

```bash
tuiml test-statistics --test friedman \
    --results '{"RandomForest": [0.95, 0.93, 0.96], "SVC": [0.91, 0.90, 0.92], "NaiveBayes": [0.94, 0.95, 0.93]}'

tuiml test-statistics --test nemenyi --results '{...}' --significance-level 0.05
```

Available tests: `friedman`, `nemenyi`, `wilcoxon`, `paired_t`, `anova`, `friedman_aligned`, `quade`.

## 5. Hyperparameter Tuning (`tuiml tune`)

Grid, random, or Bayesian search over any algorithm's parameter space.

```bash
# Exhaustive grid search
tuiml tune --algorithm RandomForestClassifier --data iris.csv --target class \
    --method grid \
    --param-grid '{"n_estimators": [50, 100, 200], "max_depth": [5, 10, null]}' \
    --cv 5

# Random search: 40 samples from the space
tuiml tune --algorithm SVC --data data.csv --target target \
    --method random \
    --param-grid '{"C": [0.1, 10.0], "kernel": ["rbf", "linear"]}' \
    --n-iter 40 --scoring f1_weighted

# Bayesian (GP-based) search
tuiml tune --algorithm XGBoostClassifier --data data.csv --target target \
    --method bayesian \
    --param-grid '{"n_estimators": [50, 500, "int"], "learning_rate": [0.01, 0.3]}' \
    --n-iterations 25
```

For `grid`, a list is the set of values to try. For `random` / `bayesian`, a two-element list is a `[low, high]` range — append `"int"` to keep it integral. The best model is persisted like any other run, so its Model ID is ready for `evaluate` or `serve`.

## 6. Discovery (`tuiml list`, `tuiml describe`)

Browse the registry and read any component's parameter schema before you use it.

### 6.1 Listing Components

```bash
# Everything in the registry
tuiml list

# One category at a time
tuiml list --category algorithm
tuiml list --category preprocessing
tuiml list --category dataset
tuiml list --category feature
tuiml list --category splitting

# Filter algorithms by type
tuiml list --type classifier
tuiml list --type regressor
tuiml list --type clusterer
tuiml list --type anomaly
tuiml list --type timeseries
```

### 6.2 Searching and Paginating

```bash
# Search by name or description
tuiml list --search forest
tuiml list --search bayes
tuiml list --type classifier --search ensemble

# Page through a long list
tuiml list --category algorithm --limit 20 --offset 40

# Agent-authored algorithms, with their run history
tuiml list --category custom --include-runs
```

### 6.3 Output Formats

```bash
# Just names, one per line (for scripting)
tuiml list --format names

# JSON output
tuiml list --format json

# Verbose table with detailed metadata
tuiml list -v
```

### 6.4 Describing a Component

```bash
# Full parameter schema - types, defaults, and bounds
tuiml describe --name RandomForestClassifier

# Works for any component kind, not just algorithms
tuiml describe --name SimpleImputer
tuiml describe --name SelectKBestSelector
tuiml describe --name iris
```

This is how you avoid guessing a hyperparameter name: read the schema, then pass exactly what it lists to `-P` or `--param-grid`.

## 7. Working with Data

Inspect, generate, and transform datasets without writing a script.

### 7.1 Register a Dataset (`tuiml upload`)

```bash
# Validate a file and get a canonical path back
tuiml upload --file-path ./my_data.csv --name customer-churn

# Inline content for small datasets
tuiml upload --content "a,b,label
1,2,yes
3,4,no" --format csv --name tiny
```

Supported formats: CSV, TSV, ARFF, Parquet, Excel (xlsx/xls), JSON, JSONL, NumPy (npy/npz).

### 7.2 Preview and Profile

```bash
# Shape, dtypes, missingness, cardinality, class distribution
tuiml profile --data iris.csv --target class

# Peek at actual rows
tuiml read-data --data iris.csv --n-rows 20
tuiml read-data --data iris.csv --mode tail --n-rows 5
tuiml read-data --data iris.csv --mode sample --n-rows 10
tuiml read-data --data iris.csv --mode indices --indices '[0, 42, 99]'
tuiml read-data --data iris.csv --columns '["sepallength", "petalwidth"]'
```

### 7.3 Generate Synthetic Data

```bash
# Clustering blobs
tuiml generate --generator Blobs --n-samples 1000 --n-clusters 5

# Classification streams
tuiml generate --generator RandomRBF --n-samples 500 --n-features 10 --n-classes 3

# Regression surfaces
tuiml generate --generator Friedman --n-samples 500 --noise 0.1

# Anything generator-specific goes in --generator-params
tuiml generate --generator Moons --n-samples 400 --generator-params '{"noise": 0.15}'
```

### 7.4 Preprocess and Select Features Standalone

```bash
# Run a preprocessing pipeline and write the result to a file
tuiml preprocess --data data.csv --target class \
    --steps '["SimpleImputer", "StandardScaler"]' \
    --output cleaned.csv

# Steps can carry parameters
tuiml preprocess --data data.csv --target class \
    --steps '[{"name": "SimpleImputer", "params": {"strategy": "median"}}, "MinMaxScaler"]'

# Or run one atomic stage: split, impute, balance, scale, encode, discretize
tuiml preprocess --data data.csv --target class --stage balance

# Feature selection on its own
tuiml select-features --data data.csv --target class --method SelectKBestSelector --k 10
tuiml select-features --data data.csv --target class --method VarianceThresholdSelector --threshold 0.01
tuiml select-features --data data.csv --target class --method CFSSelector
```

### 7.5 Plots

```bash
# Model diagnostics (need a Model ID from a previous train)
tuiml plot --plot-type confusion_matrix --model-id e9071dbe77d2 --data test.csv --target class
tuiml plot --plot-type roc_curve --model-id e9071dbe77d2 --data test.csv --target class
tuiml plot --plot-type feature_importance --model-id e9071dbe77d2
tuiml plot --plot-type tree --model-id e9071dbe77d2

# Learning curve builds its own models
tuiml plot --plot-type learning_curve --algorithm RandomForestClassifier --data iris.csv --target class

# Comparison plots take experiment results
tuiml plot --plot-type cd_diagram --experiment-results results.json
```

Run `tuiml plot --help` for the full list of plot types and which inputs each one needs.

## 8. Serving Models (`tuiml serve`)

Turn any saved model into a REST API. Because the saved file is the whole pipeline, the endpoint accepts raw rows and preprocesses them the way training did.

```bash
# Serve by file path
tuiml serve --model-path ./model.pkl --port 8000

# Or by Model ID, with a friendly name in the URL
tuiml serve --model-id e9071dbe77d2 -m my_classifier -p 9000

# Production: bind publicly, several workers
tuiml serve --model-path ./model.pkl -H 0.0.0.0 -w 4

# Copy a persisted model somewhere permanent first
tuiml save --model-id e9071dbe77d2 --destination ./models/production.joblib
```

Endpoints: `GET /health`, `GET /stats`, `GET /models`, `POST /models`, `GET /models/{id}`, `POST /models/{id}/predict`, `POST /models/{id}/predict_proba`, `POST /predict`. Interactive docs at `/docs` (Swagger) and `/redoc`.

```bash
# Check what's running, then shut it down
tuiml status
tuiml stop-server --server-id 127.0.0.1:8000
tuiml stop-server                              # omit the ID to stop everything
```

## 9. Wiring TuiML into AI Agents (`tuiml setup`)

The same commands are exposed to AI clients as MCP tools. `tuiml setup` writes the client configuration for you.

```bash
tuiml setup            # interactive Auto / Manual / Quit menu
tuiml setup -y         # configure every detected client, no prompts
tuiml setup --manual   # ask per client
tuiml setup --list     # just show what's detected, change nothing
tuiml setup --client cursor --client zed   # only these

tuiml mcp              # run the MCP server yourself (stdio), for debugging
tuiml trace            # watch tool calls live as an agent makes them
tuiml trace -n 100 --no-follow --tool tuiml_train

tuiml uninstall        # unwire every client (leaves the package installed)
```

Detected clients include Claude Desktop, Claude Code, OpenClaw, Cursor, ChatGPT Desktop, Perplexity Desktop, Codex CLI, Zed, Continue, VS Code Copilot, Windsurf, and Goose.

## 10. Agent-Authored Algorithms

With `TUIML_ALLOW_USER_ALGORITHMS=1` set, new algorithms can be written, registered, and benchmarked without leaving the terminal. Each is stored under `~/.tuiml/user_algorithms/<name>/<version>/` and re-registered at startup.

```bash
# Start from a fill-in-the-blanks template
tuiml get-skeleton --kind classifier --class-name MyAlgo

# Validate and register the finished source (--code takes the source text itself)
tuiml create-algorithm --name MyAlgo --kind classifier --version 1.0.0 \
    --code "$(cat my_algo.py)"

# It is now a first-class citizen of every other command
tuiml train -a MyAlgo -d iris.csv -t class --cv 5
tuiml experiment -a MyAlgo -a RandomForestClassifier -d iris -t class --cv 10

# Read, search, and patch the source
tuiml read-algorithm --name MyAlgo
tuiml search-source --query "def fit" --name MyAlgo
tuiml edit-algorithm --name MyAlgo --old-string "..." --new-string "..." --bump-version

# Review and clean up
tuiml list --category custom --include-runs
tuiml list-files
tuiml delete-algorithm --name MyAlgo              # every version
tuiml delete-algorithm --name MyAlgo --version 1.0.0
```

`--version` must be semver and should be bumped on every change; nothing is deleted until you say so. The bare name always resolves to the newest version, while pinned aliases (`MyAlgo_v1_0_0`) resolve to an exact one — use those to compare variants head-to-head in a single `tuiml experiment`.

Source is AST-filtered before it is accepted: no `subprocess`, `socket`, `os`, `urllib`, `requests`, `ctypes`, and no `eval` / `exec` / `open` / `input`. It must define exactly one `@classifier`- or `@regressor`-decorated class whose base type matches `--kind`.

## 11. Example: Complete CLI Workflow

A typical end-to-end session.

### Step 1: Look at the Data

```bash
tuiml profile --data iris.csv --target class
tuiml read-data --data iris.csv --n-rows 10
```

### Step 2: Find Candidate Algorithms

```bash
tuiml list --type classifier
tuiml list --search tree
tuiml describe --name RandomForestClassifier
```

### Step 3: Compare Them

```bash
tuiml experiment \
    -a C45TreeClassifier -a RandomForestClassifier -a NaiveBayesClassifier \
    -d iris.csv -t class \
    --cv 10 \
    -o comparison.md -f markdown
```

### Step 4: Tune the Winner

```bash
tuiml tune --algorithm RandomForestClassifier --data iris.csv --target class \
    --method grid \
    --param-grid '{"n_estimators": [50, 100, 200], "max_depth": [5, 10, null]}' \
    --cv 10
```

### Step 5: Train the Final Pipeline

```bash
tuiml train -a RandomForestClassifier -d iris.csv -t class \
    -p SimpleImputer -p StandardScaler \
    -P '{"n_estimators": 100}' \
    --cv 10 \
    --save-path ./model.pkl \
    -o training_results.json \
    --random-seed 42 \
    -v
```

### Step 6: Evaluate on Held-Back Data

```bash
tuiml evaluate --model-path ./model.pkl -d test.csv -t class \
    -m accuracy_score -m f1_score -m precision_score -m recall_score \
    -o test_evaluation.json
```

### Step 7: Predict and Deploy

```bash
tuiml predict --model-path ./model.pkl -d new_data.csv -o predictions.csv

tuiml serve --model-path ./model.pkl --port 8000
# ... and when you're done
tuiml stop-server
```

## 12. Scripting with the CLI

Every command accepts `--json-output`, which is what makes the CLI composable.

### 12.1 Batch Training Script

```bash
#!/bin/bash
# batch_train.sh - Train multiple models

DATASET="data.csv"
TARGET="class"
CV_FOLDS=10

mkdir -p results

for ALGO in RandomForestClassifier SVC NaiveBayesClassifier C45TreeClassifier; do
    echo "Training $ALGO..."
    tuiml train -a "$ALGO" -d "$DATASET" -t "$TARGET" \
        --cv $CV_FOLDS \
        --random-seed 42 \
        -o "results/${ALGO}_results.json"
done

echo "All models trained!"
```

### 12.2 Pipeline with JSON Output

```bash
#!/bin/bash
# Process results with jq

# Every classifier in the registry, by name
ALGORITHMS=$(tuiml list --category algorithm --type classifier --format names)

for ALGO in $ALGORITHMS; do
    ACC=$(tuiml train -a "$ALGO" -d iris.csv -t class --cv 5 --json-output \
          | jq -r '.metrics.cv_accuracy_score_mean // "N/A"')
    echo "$ALGO: $ACC"
done
```

Note the metric key: cross-validated runs report `cv_<metric>_mean` and `cv_<metric>_std`; holdout runs report the bare metric name.

### 12.3 Chaining Commands by Model ID

```bash
#!/bin/bash
set -e

# Train, and capture the Model ID from the JSON output
MODEL_ID=$(tuiml train -a RandomForestClassifier -d iris.csv -t class \
           --cv 10 --json-output | jq -r '.model_id')

echo "Trained model: $MODEL_ID"

# Everything downstream refers to that ID - no file juggling
tuiml evaluate --model-id "$MODEL_ID" -d test.csv -t class -o eval.json
tuiml plot --plot-type confusion_matrix --model-id "$MODEL_ID" -d test.csv -t class
tuiml save --model-id "$MODEL_ID" --destination ./models/production.joblib
tuiml serve --model-id "$MODEL_ID" --port 8000
```

### 12.4 Parallel Experiments

```bash
#!/bin/bash
# Run experiments in parallel using GNU parallel

DATASETS="iris.csv diabetes.csv glass.csv"
ALGOS="RandomForestClassifier SVC NaiveBayesClassifier"

mkdir -p results

parallel --jobs 4 \
    "tuiml train -a {1} -d {2} -t class --cv 10 -o results/{1}_{2/.}.json" \
    ::: $ALGOS ::: $DATASETS
```

## 13. Environment Variables

| Variable | Description | Default |
|----------|-------------|---------|
| `TUIML_ALLOW_USER_ALGORITHMS` | Enable the agent-authored algorithm commands | unset (disabled) |
| `TUIML_MCP_TRACE_FILE` | Where `tuiml trace` reads and writes tool-call traces | `~/.tuiml/logs/mcp.jsonl` |

TuiML also keeps its own state under `~/.tuiml/`: trained models in `models/`, agent-authored algorithms in `user_algorithms/`, and logs in `logs/`.

```bash
# Enable agent-authored algorithms for this shell
export TUIML_ALLOW_USER_ALGORITHMS=1
```

## 14. Command Reference

| Command | Description |
|---------|-------------|
| `tuiml train` | Train a model with preprocessing, feature selection, and evaluation |
| `tuiml predict` | Predict, get probabilities, or forecast with a saved model |
| `tuiml evaluate` | Compute metrics or a classification report on test data |
| `tuiml experiment` | Compare multiple algorithms across datasets with CV |
| `tuiml tune` | Grid, random, or Bayesian hyperparameter search |
| `tuiml test-statistics` | Friedman / Nemenyi / Wilcoxon / paired-t / ANOVA / Quade tests |
| `tuiml upload` | Register and validate a dataset file or inline content |
| `tuiml read-data` | Preview rows (head / tail / sample / indices) |
| `tuiml profile` | Shape, dtypes, missingness, cardinality, class balance |
| `tuiml generate` | Generate synthetic datasets |
| `tuiml preprocess` | Run preprocessing steps or one atomic stage standalone |
| `tuiml select-features` | Run feature selection standalone |
| `tuiml plot` | Confusion matrix, ROC, PR, learning curve, tree, CD diagram, … |
| `tuiml list` | Browse and search the registry |
| `tuiml describe` | Parameter schema for any component |
| `tuiml serve` | Start a REST API for a model |
| `tuiml save` | Copy a persisted model to a path you choose |
| `tuiml stop-server` | Stop one server, or all of them |
| `tuiml status` | Show running servers and MCP processes |
| `tuiml setup` / `uninstall` | Wire / unwire TuiML into AI clients |
| `tuiml mcp` | Run the MCP server over stdio |
| `tuiml trace` | View or follow MCP tool-call traces |
| `tuiml info` / `update` / `restart` | Version and install management |
| `tuiml get-skeleton` | Template for a new algorithm |
| `tuiml create-algorithm` | Validate and register new algorithm source |
| `tuiml read-algorithm` / `edit-algorithm` / `delete-algorithm` | Manage user algorithms |
| `tuiml list-files` / `search-source` | Locate and grep algorithm source |

Use `tuiml <command> --help` for detailed options.